<a href="https://colab.research.google.com/github/quarcs-lab/gdo4bolivia/blob/main/notebooks/esda_dependence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spatial distribution and spatial dependence of poverty in Bolivia

Part of the [GeoDevelopment Observatory of Bolivia](https://quarcs-lab.github.io/gdo4bolivia/).

How is multidimensional poverty spread across Bolivia's 343 municipalities and indigenous autonomies, how much does a municipality resemble its neighbours, and did either change between 2012 and 2024? This notebook uses the Multidimensional Poverty Index (MPI; 0–1, higher = poorer) of the *Atlas de los ODS en Bolivia 2026* (SDSN Bolivia), computed from the population censuses of 2012 and 2024, and the functions of the [`geo4bolivia`](https://quarcs-lab.github.io/geo4bolivia/) package:

1. **Load the data** (`geo.load`)
2. **Spatial distribution**: two ways of classifying a map, box-plot breaks and Fisher-Jenks natural breaks, in both periods (`geo.distribution`, `geo.map` with `bins=` and `scheme=`)
3. **Spatial dependence**: spatial weights, the spatial lag, global Moran's I, and local clusters and outliers (LISA) at two significance levels (`geo.weights`, `geo.moran`, `geo.lisa` with `alpha=`)

Two steps have no function in the package and call the underlying libraries directly: the box-plot breaks (`mapclassify`) and the spatial lag in the units of the index (`libpysal`). Both are then mapped with `geo.map`.

## 0. Setup

In Google Colab the next cell installs the package (a few seconds); it brings the spatial-analysis libraries used here (libpysal, esda, mapclassify, plotly). Locally, `uv sync` has installed it already.

In [1]:
import importlib.util

missing = [p for p in ["geo4bolivia"] if importlib.util.find_spec(p) is None]
if missing:
    pkgs = " ".join(missing)
    !pip install -q {pkgs}

In [2]:
import warnings
warnings.filterwarnings("ignore")    # library notices; the text explains the ones that matter

import pandas as pd
import mapclassify as mc
from libpysal.weights import lag_spatial

import geo4bolivia as geo

YEARS = [2012, 2024]
MPI = "sdg1_2_mpi"                   # Multidimensional Poverty Index (0-1, higher = poorer)

## 1. Load the data

`geo.load()` returns a panel with one row per municipality (`geo_id`) and year. The Atlas gives the MPI for two periods, stored as the years 2012 and 2024. `groups=` adds the name of each municipality and of its department.

In [3]:
mpi = geo.load("sdg01", variables=MPI, groups=["department"])
print(mpi["geo_id"].nunique(), "municipalities and indigenous autonomies")
mpi.head()

343 municipalities and indigenous autonomies


,geo_id,name,year,department,sdg1_2_mpi
0,BO-01-01-01,Azurduy,2012,Chuquisaca,0.512
1,BO-01-01-01,Azurduy,2024,Chuquisaca,0.187
2,BO-01-01-02,Tarvita,2012,Chuquisaca,0.452
3,BO-01-01-02,Tarvita,2024,Chuquisaca,0.133
4,BO-01-02-01,Villa Serrano,2012,Chuquisaca,0.267


## 2. Spatial distribution

How we cut a continuous variable into classes changes what a map shows. This section compares two classifications, each computed separately for each period, so the classes describe where poverty is relatively high or low in that period.

### Box-plot breaks

Box-plot breaks follow the box plot: the classes are the quartiles, and values beyond 1.5 times the interquartile range from the box are flagged as outliers. `geo.distribution()` draws the box plots of both periods:

In [4]:
geo.distribution(mpi, MPI, kind="box")

Between 2012 and 2024, the mean of **Multidimensional poverty index (0-1)** fell from 0.234 to 0.0633, and the median went from 0.227 to 0.053.

The spread across units narrowed: the standard deviation went from 0.106 to 0.0441, and the interquartile range (the middle half of the units) from 0.153 to 0.053.

**Multidimensional poverty index (0-1)** in 2024 averages 0.0633 across 343 units (median 0.053, standard deviation 0.0441). Half of the units lie between 0.03 and 0.083, and the full range runs from 0.003 to 0.296. The distribution is strongly right-skewed: a tail of high values pulls the mean above the median (skewness 1.44).

In each box, the line is the median and the box holds the middle half of the units; points beyond the whiskers are unusually high or low units.

_These are associations, not causal effects. A causal reading needs a research design._

The package has no function for box-plot breaks, so we take them from `mapclassify`:

In [5]:
box = {year: mc.BoxPlot(mpi.loc[mpi["year"] == year, MPI]) for year in YEARS}
for year in YEARS:
    print(year, box[year], "\n")

2012 BoxPlot

   Interval      Count
----------------------
( -inf, -0.08] |     0
(-0.08,  0.15] |    87
( 0.15,  0.23] |    86
( 0.23,  0.31] |    84
( 0.31,  0.54] |    84
( 0.54,  0.56] |     2 

2024 BoxPlot

   Interval      Count
----------------------
( -inf, -0.05] |     0
(-0.05,  0.03] |    88
( 0.03,  0.05] |    84
( 0.05,  0.08] |    87
( 0.08,  0.16] |    72
( 0.16,  0.30] |    12 



`geo.map()` accepts explicit class limits as `bins=`. The lower fence lies below zero in both periods, so no municipality is a low outlier and we leave that limit out. The last class of each map holds the high outliers.

In [6]:
for year in YEARS:
    lowest = mpi.loc[mpi["year"] == year, MPI].min()
    limits = [float(b) for b in box[year].bins[:-1] if b > lowest]
    geo.map(mpi, MPI, year=year, bins=limits, title=f"Box-plot breaks, {year}").show()

The high outliers of each period, and how much their MPI changed since 2012 (`geo.change()` gives one row per municipality):

In [7]:
chg = geo.change(mpi, MPI)
fence = {year: box[year].bins[4] for year in YEARS}                 # the upper fence
for year in YEARS:
    chg[f"outlier_{year}"] = chg[f"{MPI}_{year}"] > fence[year]
print("High outliers:", {year: int(chg[f"outlier_{year}"].sum()) for year in YEARS})
print(f"Median change of the MPI, 2012-2024: {chg[f'{MPI}_pct_change'].median():.0f}%")
cols = ["name", "department", f"{MPI}_2012", f"{MPI}_2024", f"{MPI}_pct_change", "outlier_2012"]
chg[chg["outlier_2024"]].sort_values(f"{MPI}_2024", ascending=False)[cols].round(3)

High outliers: {2012: 2, 2024: 12}
Median change of the MPI, 2012-2024: -76%


,name,department,sdg1_2_mpi_2012,sdg1_2_mpi_2024,sdg1_2_mpi_pct_change,outlier_2012
63,TIOC-Raqaypampa,Cochabamba,0.563,0.296,-47.425,True
12,Poroma,Chuquisaca,0.541,0.223,-58.780,True
240,San Pedro,Potosí,0.496,0.213,-57.056,False
189,El Choro,Oruro,0.443,0.203,-54.176,False
62,Vila Vila,Cochabamba,0.461,0.200,-56.616,False
9,Incahuasi,Chuquisaca,0.462,0.191,-58.658,False
0,Azurduy,Chuquisaca,0.512,0.187,-63.477,False
256,Ckochas,Potosí,0.475,0.182,-61.684,False
229,San Pedro,Pando,0.472,0.177,-62.500,False
249,Tacobamba,Potosí,0.483,0.171,-64.596,False


The upper quartile of each period (the two highest classes of each map), by department:

In [8]:
q3 = {year: box[year].bins[3] for year in YEARS}
upper = mpi[mpi[MPI] > mpi["year"].map(q3)]
pd.crosstab(upper["department"], upper["year"], margins=True, margins_name="Total")

year,2012,2024,Total
department,,,
Beni,5,7,12
Chuquisaca,17,12,29
Cochabamba,13,14,27
La Paz,13,19,32
Oruro,11,10,21
Pando,9,3,12
Potosí,17,19,36
Santa Cruz,1,0,1
Total,86,84,170


In 2012 the median MPI is 0.227, and the middle half of the municipalities lies between 0.153 and 0.306. The lower fence is below zero, so no municipality is a low outlier, while two are high outliers above 0.54: TIOC-Raqaypampa (Cochabamba, 0.563) and Poroma (Chuquisaca, 0.541).

By 2024 the whole distribution has shifted down and narrowed: the median is 0.053 and the middle half lies between 0.030 and 0.083. Again there are no low outliers, but now 12 units are high outliers above 0.16, led again by TIOC-Raqaypampa (0.296) and Poroma (0.223), followed by San Pedro (Potosí), El Choro (Oruro) and Vila Vila (Cochabamba).

The median municipality cut its MPI by 76% between the two periods, while the 12 high outliers of 2024 cut theirs by 47% to 65%. The poorest units progressed more slowly in relative terms, which is why they stand further from the rest of the distribution in 2024 than in 2012.

On both maps the upper quartile is mostly in Potosí, Chuquisaca, Cochabamba, La Paz and Oruro, which hold 71 of its 86 units in 2012 and 74 of its 84 in 2024. Pando had 9 units in the upper quartile in 2012 and has 3 in 2024; La Paz went from 13 to 19. Santa Cruz has one unit in 2012 and none in 2024, and Tarija none in either period.

### Fisher-Jenks breaks

The Fisher-Jenks algorithm chooses the breaks that make the classes as homogeneous as possible (it minimises the variance within classes). `geo.map()` computes them with `scheme="fisher_jenks"`; here, three classes per period. A classed map keeps its class limits in `fig.layout.meta["bins"]`, which we use to count the municipalities in each class.

In [9]:
classes = {}
for year in YEARS:
    fig = geo.map(mpi, MPI, year=year, scheme="fisher_jenks", k=3,
                  title=f"Three natural breaks, {year}")
    one = mpi[mpi["year"] == year]
    classes[year] = pd.cut(one[MPI], fig.layout.meta["bins"], include_lowest=True,
                           labels=["low", "middle", "high"])
    print(year, "class limits:", fig.layout.meta["bins"],
          "| municipalities per class:", classes[year].value_counts(sort=False).tolist())
    fig.show()

2012 class limits: [0.018, 0.179, 0.308, 0.563] | municipalities per class: [112, 148, 83]


2024 class limits: [0.003, 0.052, 0.11, 0.296] | municipalities per class: [166, 132, 45]


The high class of each period, by department:

In [10]:
high = mpi[pd.concat(classes.values()) == "high"]
pd.crosstab(high["department"], high["year"], margins=True, margins_name="Total")

year,2012,2024,Total
department,,,
Beni,5,2,7
Chuquisaca,16,9,25
Cochabamba,13,9,22
La Paz,12,4,16
Oruro,10,9,19
Pando,9,1,10
Potosí,17,11,28
Santa Cruz,1,0,1
Total,83,45,128


In 2012 natural breaks put 112 municipalities in the low class (up to 0.18), 148 in the middle class (0.18 to 0.31) and 83 in the high class (above 0.31). In 2024 they put 166 in the low class (up to 0.05), 132 in the middle (0.05 to 0.11) and 45 in the high class (above 0.11).

Box-plot breaks give each quartile about 85 units by construction. Natural breaks let the data decide the size of the classes, and they single out the group of very poor municipalities. In 2024 that group is smaller and more concentrated: 38 of its 45 units are in Potosí (11), Chuquisaca (9), Cochabamba (9) and Oruro (9). Pando had 9 units in the high class in 2012 and has 1 in 2024; La Paz went from 12 to 4.

## 3. Spatial dependence

- To what extent is the poverty of a municipality similar to that of its neighbours?
- To what extent is there an overall pattern of clustering, and did it weaken as poverty fell?
- Where are the statistically significant spatial clusters?
- Where are the statistically significant spatial outliers?

### Spatial connectivity

Each municipality is connected to its 6 nearest neighbours (by the distance between centroids), on all 343 units. `geo.weights()` builds the weights row-standardised, so each neighbour counts 1/6. With nearest neighbours no unit is left without neighbours. The same weights serve both periods. The companion notebook `esda` compares them with the package's default, queen contiguity.

In [11]:
w = geo.weights("municipality", method="knn", k=6)
links = geo.connectivity("municipality", w)
print(f"{links.n_units} units, {links.mean_neighbors:.0f} neighbours each;",
      f"islands: {len(links.islands)}; connected components: {links.n_components}")

343 units, 6 neighbours each; islands: 0; connected components: 1


### The spatial lag

The **spatial lag** of the MPI is the average poverty of each municipality's neighbours. The package computes it inside `geo.moran()`, in standard deviations from the mean. To have it in the units of the index we call `libpysal` directly, on the values of each year in the order of the weights, and keep it as a column of the panel:

In [12]:
parts = []
for year in YEARS:
    one = mpi[mpi["year"] == year].set_index("geo_id").loc[w.id_order]    # in the order of the weights
    parts.append(one.assign(mpi_lag=lag_spatial(w, one[MPI].to_numpy())).reset_index())
mpi = pd.concat(parts, ignore_index=True)
mpi[mpi["name"].isin(["Azurduy", "Sucre", "Nuestra Señora de La Paz"])].round(3)

,geo_id,name,year,department,sdg1_2_mpi,mpi_lag
0,BO-01-01-01,Azurduy,2012,Chuquisaca,0.512,0.377
13,BO-01-06-02,Sucre,2012,Chuquisaca,0.071,0.372
161,BO-04-17-04,Nuestra Señora de La Paz,2012,La Paz,0.018,0.129
343,BO-01-01-01,Azurduy,2024,Chuquisaca,0.187,0.122
356,BO-01-06-02,Sucre,2024,Chuquisaca,0.014,0.115
504,BO-04-17-04,Nuestra Señora de La Paz,2024,La Paz,0.003,0.030


Azurduy (Chuquisaca) had an MPI of 0.512 in 2012 and its six neighbours 0.377 on average; in 2024, 0.187 and 0.122. Its neighbours are poor, but less so than Azurduy itself. Sucre is the opposite case: an MPI of 0.014 in 2024 among neighbours that average 0.115.

With the lag as a column, `geo.map()` draws it like any other variable. The map of 2024 is a smoothed version of the map of poverty: each municipality is coloured by the average of its six neighbours.

In [13]:
geo.map(mpi, "mpi_lag", year=2024,
        title="Spatial lag of the MPI (average of the 6 nearest neighbours), 2024")

### Global dependence

The Moran scatterplot plots each municipality's poverty against its spatial lag, both in standard deviations from the mean of the year. **Moran's I** is the slope of the line through the cloud. Its pseudo p-value compares it with 999 random reshuffles of the values across the map.

In [14]:
moran = {year: geo.moran(mpi, MPI, year=year, w=w) for year in YEARS}
moran[2012]

Global Moran's I for **Multidimensional poverty index (0-1)** in 2012 is 0.448, against an expectation of -0.00292 under spatial randomness — statistically significant at the 1% level (pseudo p = 0.001 from 999 permutations, under 6-nearest-neighbor (metric centroids), row-standardized, n=343).

The dependence is **positive**: similar values cluster in space — high values sit next to high values and low next to low, so the map shows contiguous patches rather than a random scatter.

245 of 343 units (71%) fall in the clustering quadrants of the scatter (High-High or Low-Low); the rest are surrounded by neighbors unlike themselves. The slope of the fitted line equals Moran's I under row-standardized weights, so a steeper line means stronger clustering.

_These are associations, not causal effects. A causal reading needs a research design._

In [15]:
moran[2024]

Global Moran's I for **Multidimensional poverty index (0-1)** in 2024 is 0.388, against an expectation of -0.00292 under spatial randomness — statistically significant at the 1% level (pseudo p = 0.001 from 999 permutations, under 6-nearest-neighbor (metric centroids), row-standardized, n=343).

The dependence is **positive**: similar values cluster in space — high values sit next to high values and low next to low, so the map shows contiguous patches rather than a random scatter.

256 of 343 units (75%) fall in the clustering quadrants of the scatter (High-High or Low-Low); the rest are surrounded by neighbors unlike themselves. The slope of the fitted line equals Moran's I under row-standardized weights, so a steeper line means stronger clustering.

_These are associations, not causal effects. A causal reading needs a research design._

The same slope appears in the units of the index. `geo.scatter()` fits the least-squares line of the lag computed above on the MPI:

In [16]:
for year in YEARS:
    line, m = geo.scatter(mpi, MPI, "mpi_lag", year=year), moran[year]
    print(f"{year}: Moran's I = {m.moran_i:.3f} (pseudo p-value {m.p_sim:.3f});",
          f"slope of the lag on the MPI = {line.slope:.3f}")

2012: Moran's I = 0.448 (pseudo p-value 0.001); slope of the lag on the MPI = 0.448
2024: Moran's I = 0.388 (pseudo p-value 0.001); slope of the lag on the MPI = 0.388


Moran's I is 0.448 in 2012 and 0.388 in 2024, both with a pseudo p-value of 0.001, the smallest possible with 999 permutations: no random reshuffle of the values across the map produced as much clustering. With row-standardised weights, Moran's I is exactly the slope of the lag on the variable, in standard deviations or in the units of the index (0.448 and 0.388). In 2012, a municipality whose MPI is 0.1 above the mean has neighbours whose average MPI is, on average, about 0.045 above the mean. Spatial dependence weakened somewhat as poverty fell, but it is still strong in 2024.

### Local dependence

Local Moran's I (LISA) tests, municipality by municipality, whether its value and its neighbours' are more alike (clusters) or more different (outliers) than a random arrangement would give. `alpha=` sets the significance level below which a municipality is assigned to a type. Here we use 10%, looser than the default of 5%. "High" and "low" are relative to each period's mean.

In [17]:
lisa = {year: geo.lisa(mpi, MPI, year=year, w=w, alpha=0.10) for year in YEARS}
lisa[2012]

Local Moran statistics (LISA) locate *where* **Multidimensional poverty index (0-1)** in 2012 clusters or stands out, under 6-nearest-neighbor (metric centroids), row-standardized, n=343. The accompanying global Moran's I is 0.448 (pseudo p = 0.001), consistent with overall clustering of similar values.

At the 0.1 significance level, 157 of 343 units show significant local association: **52 High-High** hot spots (high values surrounded by high neighbors) and **83 Low-Low** cold spots (low surrounded by low) mark clustering, while **10 High-Low** and **12 Low-High** units are spatial outliers that break with their surroundings. The remaining 186 units are not significant — their local pattern is compatible with randomness.

LISA pseudo p-values are computed unit by unit without a multiple-testing adjustment, so treat borderline clusters cautiously and read the map as descriptive of where dependence concentrates.

_These are associations, not causal effects. A causal reading needs a research design._

In [18]:
lisa[2024]

Local Moran statistics (LISA) locate *where* **Multidimensional poverty index (0-1)** in 2024 clusters or stands out, under 6-nearest-neighbor (metric centroids), row-standardized, n=343. The accompanying global Moran's I is 0.388 (pseudo p = 0.001), consistent with overall clustering of similar values.

At the 0.1 significance level, 143 of 343 units show significant local association: **36 High-High** hot spots (high values surrounded by high neighbors) and **85 Low-Low** cold spots (low surrounded by low) mark clustering, while **9 High-Low** and **13 Low-High** units are spatial outliers that break with their surroundings. The remaining 200 units are not significant — their local pattern is compatible with randomness.

LISA pseudo p-values are computed unit by unit without a multiple-testing adjustment, so treat borderline clusters cautiously and read the map as descriptive of where dependence concentrates.

_These are associations, not causal effects. A causal reading needs a research design._

The Moran scatterplot of 2024 coloured by type:

In [19]:
lisa[2024].fig_scatter

How many municipalities fall in each type, at the 10% and at the stricter 5% level, in each period?

In [20]:
order = ["High-High", "Low-Low", "High-Low", "Low-High", "Not significant"]
counts = {}
for year in YEARS:
    strict = geo.lisa(mpi, MPI, year=year, w=w, alpha=0.05)
    counts[(year, "p < 0.10")] = lisa[year].df["cluster"].value_counts()
    counts[(year, "p < 0.05")] = strict.df["cluster"].value_counts()
pd.DataFrame(counts).reindex(order)

2012              2024         
                p < 0.10 p < 0.05 p < 0.10 p < 0.05
cluster                                            
High-High             52       43       36       27
Low-Low               83       70       85       71
High-Low              10        7        9        3
Low-High              12        7       13        6
Not significant      186      216      200      236

The types at the 10% level by department:

In [21]:
clusters = pd.concat([mpi[mpi["year"] == year].merge(lisa[year].df[["geo_id", "cluster"]])
                      for year in YEARS])
(pd.crosstab(clusters["department"], [clusters["year"], clusters["cluster"]])
   .reindex(columns=order[:4], level=1))

year            2012                                2024                   \
cluster    High-High Low-Low High-Low Low-High High-High Low-Low High-Low   
department                                                                  
Beni               2       0        0        2         3       0        0   
Chuquisaca        15       1        0        3         9       3        1   
Cochabamba         8      21        1        0         5      12        2   
La Paz             5      19        2        3         1      16        2   
Oruro              2       3        1        1         4       2        4   
Pando              8       0        0        3         2       0        0   
Potosí            12       0        1        0        12       2        0   
Santa Cruz         0      31        5        0         0      41        0   
Tarija             0       8        0        0         0       9        0   

year                 
cluster    Low-High  
department           
Beni              1  
Chuquisaca        3  
Cochabamba        1  
La Paz            3  
Oruro             0  
Pando             3  
Potosí            2  
Santa Cruz        0  
Tarija            0

At the 10% level, 157 municipalities belong to a significant type in 2012 (30 more than at 5%) and 143 in 2024 (36 more than at 5%):

- **Poverty hot spots (High-High) fell from 52 to 36.** In 2012 they were in Chuquisaca (15), Potosí (12), Pando (8), Cochabamba (8) and La Paz (5), with 2 each in Beni and Oruro. In 2024 they are in Potosí (12), Chuquisaca (9), Cochabamba (5), Oruro (4) and Beni (3), while Pando keeps 2 and La Paz 1.
- **Clusters of low poverty (Low-Low) held steady, 83 and 85**, but shifted east: Santa Cruz went from 31 to 41 and Tarija from 8 to 9, while Cochabamba fell from 21 to 12 and La Paz from 19 to 16.
- **Outliers: 22 in 2012 and 22 in 2024**: 10, then 9, poorer municipalities with less poor neighbours (High-Low), and 12, then 13, less poor municipalities with poorer neighbours (Low-High).

A looser threshold flags more municipalities but also more false positives: with 343 tests at 10%, about 34 would be significant by chance even with no spatial pattern at all. The core clusters are the same at both levels: the hot spots in Potosí, Chuquisaca and Cochabamba, and the clusters of low poverty in Santa Cruz, Cochabamba, La Paz and Tarija.

## Summary

- The MPI fell sharply between 2012 and 2024 (median 0.227 to 0.053). Box-plot breaks flag 2 high outliers in 2012 and 12 in 2024, led in both by TIOC-Raqaypampa and Poroma; three Fisher-Jenks classes isolate 83 very poor units in 2012 (above 0.31) and 45 in 2024 (above 0.11).
- Poverty is strongly spatially dependent in both periods: Moran's I = 0.448 in 2012 and 0.388 in 2024 (pseudo p-value 0.001) with 6-nearest-neighbour weights on all 343 units.
- At the 10% level, LISA finds 52 poverty hot spots in 2012 and 36 in 2024, now concentrated in Potosí, Chuquisaca and Cochabamba, and 83 and 85 clusters of low poverty, increasingly in Santa Cruz.
- These results describe where poverty is and how it is arranged in space; they are associations, not causes.

Function pages: [`distribution`](https://quarcs-lab.github.io/geo4bolivia/reference/distribution.html), [`map`](https://quarcs-lab.github.io/geo4bolivia/reference/map.html), [`weights`](https://quarcs-lab.github.io/geo4bolivia/reference/weights.html), [`moran`](https://quarcs-lab.github.io/geo4bolivia/reference/moran.html), [`lisa`](https://quarcs-lab.github.io/geo4bolivia/reference/lisa.html).

The companion notebook [`esda`](https://colab.research.google.com/github/quarcs-lab/gdo4bolivia/blob/main/notebooks/esda.ipynb) lists the clusters and outliers at the 5% level, compares the weights with queen contiguity and counts how units moved between types.